In [ ]:
import torch
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.datasets as datasets
import torchvision.models as models
import torchvision.transforms as transforms
import glob
import PIL.Image
import os
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

In [ ]:
# 1. 기본 설정 및 경로 (데이터셋 폴더가 코드와 같은 위치에 있다고 가정)
DATASET_DIR = 'dataset_xy'
BEST_MODEL_PATH = 'best_model_xy.pth'
center = 224./2.

# 2. 헬퍼 함수
def get_x(path):
    """파일명에서 x 좌표 추출 (000_000 형식 대응)"""
    return (float(int(path[3:6]))/100.) - (center/100.)

def get_y(path):
    """파일명에서 y 좌표 추출"""
    return (float(int(path[7:10])) - 50.0) / 50.0

def get_lr(optimizer):
    for param_group in optimizer.param_groups:
        return param_group['lr']

데이터셋 클래스 정의

In [ ]:
class XYDataset(torch.utils.data.Dataset):
    def __init__(self, directory, random_hflips=False):
        self.directory = directory
        self.random_hflips = random_hflips
        self.image_paths = glob.glob(os.path.join(self.directory, '*.jpg'))
        self.color_jitter = transforms.ColorJitter(0.3, 0.3, 0.3, 0.3)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        image = PIL.Image.open(image_path)
        x = float(get_x(os.path.basename(image_path)))
        y = float(get_y(os.path.basename(image_path)))

        if self.random_hflips and float(np.random.rand(1)) > 0.5:
            image = transforms.functional.hflip(image)
            x = -x

        image = self.color_jitter(image)
        image = transforms.functional.to_tensor(image)
        image = image.numpy()[::-1].copy()
        image = torch.from_numpy(image)
        image = transforms.functional.normalize(image, [0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

        return image, torch.tensor([x, y]).float()

# 4. 데이터 로더 설정
# 윈도우 환경에서는 num_workers를 0으로 설정하는 것이 가장 안전합니다.
dataset = XYDataset(DATASET_DIR, random_hflips=True)
test_percent = 0.2
num_test = int(test_percent * len(dataset))
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [len(dataset) - num_test, num_test])

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0 
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0
)

Model

In [ ]:
model = models.resnet18(pretrained=True)
model.fc = torch.nn.Linear(512, 2)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f"Using device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

main 함수

In [ ]:
NUM_EPOCHS = 90
best_loss = 1e9
optimizer = optim.Adam(model.parameters(), lr=0.001)

epoch_list, train_loss_list, test_loss_list = [], [], []

for epoch in range(NUM_EPOCHS):
    model.train()
    train_loss = 0.0
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = F.mse_loss(outputs, labels)
        train_loss += float(loss)
        loss.backward()
        optimizer.step()
    train_loss /= len(train_loader)

    model.eval()
    test_loss = 0.0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = F.mse_loss(outputs, labels)
            test_loss += float(loss)
    test_loss /= len(test_loader)

    print(f'Epoch {epoch+1}: Train Loss {train_loss:.6f}, Test Loss {test_loss:.6f}')

    epoch_list.append(epoch)
    train_loss_list.append(train_loss)
    test_loss_list.append(test_loss)
    
    if test_loss < best_loss:
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        best_loss = test_loss
        print(f"*** Best model saved at epoch {epoch+1}")

결과 그래프 출력

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(epoch_list, train_loss_list, '-b', label='Train Loss')
plt.plot(epoch_list, test_loss_list, '-r', label='Test Loss')
plt.xlabel("Epochs")
plt.ylabel("Loss (MSE)")
plt.legend()
plt.title("Lane Following Training Results")
plt.show()